# Human Action Classifier
**Schuldt et al. 2004. Recognizing Human Actions: A Local SVM Approach**

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import re
import csv

from pathlib import Path
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from joblib import dump

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

mps


### 1. Data Loading

In [3]:
PREFIX = "0229-"
DESCRIPTION = "With 5-Fold Subject-Based Cross-Validation, Video Transformation (Horizontal Flip), and Empty Scene"
K = 800
ACTIONS = ["boxing", "handclapping", "handwaving", "jogging", "running", "walking", "empty"]
action_to_id = {a: i for i, a in enumerate(ACTIONS)}

VIDEO_ID_PATTERN = re.compile(r"person(?P<person>\d+)_(?P<action>[a-z]+)_d(?P<scene>\d+)")

def index_data(csv_file="data/info.csv", clips_root="data/transform"):
    samples = []

    with open(csv_file, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            video_id = row["video_id"]
            target_video = row["target_video"]
            action = row["label"]

            match = VIDEO_ID_PATTERN.match(video_id)
            if match is None or action not in action_to_id:
                continue

            clip_path = Path(clips_root) / target_video
            if not clip_path.exists():
                continue    # Skip missing clips (safe for partial generation)

            samples.append({
                "path": clip_path,
                "label": action_to_id[action],
                "action": action,
                "person": int(match.group("person")),
                "scene": int(match.group("scene")),
                "segment_id": int(row["segment_id"]),
                "video_id": video_id,
                "start_frame": int(row["start_frame"]),
                "end_frame": int(row["end_frame"]),
            })

    return samples

In [4]:
# Import all data
samples = index_data()
print("Total:", Counter(s["action"] for s in samples))

# Train/test split - use persons 1-20 for k-fold CV, 21+ for final test
trainval_samples = [s for s in samples if s["person"] <= 21]
test_samples = [s for s in samples if s["person"] > 21]
print("Train + Validation:", len(trainval_samples), "Test:", len(test_samples))

Total: Counter({'empty': 1918, 'jogging': 800, 'running': 800, 'walking': 800, 'handwaving': 796, 'boxing': 794, 'handclapping': 792})
Train + Validation: 5624 Test: 1076


### 2. Data Processing
#### 2.1. Transform Video To Image Sequence

In [5]:
def load_video(path, resize=(160, 120)):
    cap = cv2.VideoCapture(str(path))
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, resize)
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    cap.release()
    video = np.stack(frames)   # (T, H, W)
    return torch.from_numpy(video)

#### 2.2. Construct Gaussian Scale Space
$$L(· , \sigma^2, \tau^2) = f \times g(· , \sigma^2, \tau^2)$$

In [6]:
def gaussian_1d(kernel_size, sigma):
    x = torch.arange(kernel_size) - kernel_size // 2
    g = torch.exp(-(x ** 2) / (2 * (sigma ** 2)))
    return g / g.sum()

def gaussian_blur_3d(video, sigma, tau):
    # spatial blur
    g_xy = gaussian_1d(7, sigma)
    g_xy = g_xy[None, None, :, None] * g_xy[None, None, None, :]
    g_xy = g_xy.to(video.device)
    video = F.conv2d(video.unsqueeze(1), g_xy, padding=3).squeeze(1)

    # temporal blur
    g_t = gaussian_1d(7, tau)[None, None, :, None, None]
    g_t = g_t.to(video.device)
    video = F.conv3d(video.unsqueeze(0), g_t, padding=(3,0,0)).squeeze(0)

    return video

### 3. Feature Extraction
#### 3.1. Compute Second-moment Matrix From Spatio-temporal Gradients

$$\nabla L = (L_x, L_y, L_t)^T$$
$$\mu(·; \sigma^2, \tau^2) = g(·; s\sigma^2, s\tau^2) \times (\nabla L(\nabla L)^T)$$

In [7]:
def gradients_3d(L):
	Lx = L[:, :, 2:] - L[:, :, :-2]
	Ly = L[:, 2:, :] - L[:, :-2, :]
	Lt = L[2:, :, :] - L[:-2, :, :]

	T = min(Lx.shape[0], Ly.shape[0], Lt.shape[0])
	H = min(Lx.shape[1], Ly.shape[1], Lt.shape[1])
	W = min(Lx.shape[2], Ly.shape[2], Lt.shape[2])

	return Lx[:T, :H, :W], Ly[:T, :H, :W], Lt[:T, :H, :W]

def second_moment_matrix(Lx, Ly, Lt, sigma=2.0, tau=1.5):
    J_xx = Lx * Lx
    J_xy = Lx * Ly
    J_xt = Lx * Lt
    J_yy = Ly * Ly
    J_yt = Ly * Lt
    J_tt = Lt * Lt

    def smooth(x):
        return gaussian_blur_3d(x, sigma, tau)

    return smooth(J_xx), smooth(J_xy), smooth(J_xt), smooth(J_yy), smooth(J_yt), smooth(J_tt)

#### 3.2. Detect Interest Point Using Harris Response

$$H = det(\mu) - k * trace^3(\mu)$$

In [8]:
def harris_response(J, k=0.005):
    J_xx, J_xy, J_xt, J_yy, J_yt, J_tt = J

    det = (
        J_xx * (J_yy * J_tt - J_yt ** 2)
        - J_xy * (J_xy * J_tt - J_xt * J_yt)
        + J_xt * (J_xy * J_yt - J_xt * J_yy)
    )

    trace = J_xx + J_yy + J_tt
    return det - k * trace ** 3

def detect_interest_points(H, threshold_ratio=0.01):
    threshold = threshold_ratio * H.max()
    points = torch.nonzero(H > threshold)
    return points

#### 3.3. Extract Spatio-temporal Descriptors

$$\bold{l} = (L_x, L_y, L_t, L_{xx}, ..., L_{tttt})$$

In [9]:
def extract_jet(L, x, y, t):
    Lc = L[t, y, x]

    # first order
    Lx  = L[t, y, x+1] - L[t, y, x-1]
    Ly  = L[t, y+1, x] - L[t, y-1, x]
    Lt  = L[t+1, y, x] - L[t-1, y, x]

    # second order
    Lxx = L[t, y, x+1] - 2*Lc + L[t, y, x-1]
    Lyy = L[t, y+1, x] - 2*Lc + L[t, y-1, x]
    Ltt = L[t+1, y, x] - 2*Lc + L[t-1, y, x]

    return Lx, Ly, Lt, Lxx, Lyy, Ltt

def extract_descriptors(video, sigma=1.5, tau=1.0):
    L = gaussian_blur_3d(video, sigma, tau)
    Lx, Ly, Lt = gradients_3d(L)
    J = second_moment_matrix(Lx, Ly, Lt, 2*sigma, 2*tau)
    H = harris_response(J)
    t, y, x = detect_interest_points(H).T

    mask = (
        (t > 1) & (y > 1) & (x > 1) &
        (t < L.shape[0] - 2) &
        (y < L.shape[1] - 2) &
        (x < L.shape[2] - 2)
    )
    t, y, x = t[mask], y[mask], x[mask]
    if t.numel() == 0:
        return torch.empty((0, 6), device=L.device)
    
    sigma2 = sigma * sigma
    tau2 = tau * tau
    Lx, Ly, Lt, Lxx, Lyy, Ltt = extract_jet(L, x, y, t)
    desc = torch.stack([sigma * Lx, sigma * Ly, tau * Lt, sigma2 * Lxx, sigma2 * Lyy, tau2 * Ltt], dim=1)

    return desc

#### 3.4. Build Visual Vocabulary

In [10]:
def extract_all_descriptors(samples):
    # Extract descriptors for all samples
    all_desc = []

    for s in samples:
        video = load_video(s["path"])
        video = video.to(device)
        desc = extract_descriptors(video)
        all_desc.append(desc)

    return all_desc

def build_vocabulary(descriptors, K=400, max_samples=100_000):
    # Build vocabulary from pre-extracted TRAINING descriptors
    X = torch.cat(descriptors).cpu().numpy()

    if len(X) > max_samples:
        idx = np.random.choice(len(X), max_samples, replace=False)
        X = X[idx]

    kmeans = MiniBatchKMeans(n_clusters=K, batch_size=4096, random_state=0)
    kmeans.fit(X)
    return kmeans

In [ ]:
# Extract descriptors once for train+val and test
print("Extracting descriptors for train+val dataset...")
trainval_descriptors = extract_all_descriptors(trainval_samples)

print("Extracting descriptors for test dataset...")
test_descriptors = extract_all_descriptors(test_samples)

Extracting descriptors for train+val dataset...


Extracting descriptors for test dataset...


#### 3.5. Encode Video as Histogram

In [16]:
def encode_descriptors(desc, kmeans, K):
    # Encode pre-extracted descriptors as histogram
    if desc.numel() == 0:
        return torch.zeros(K)

    labels = kmeans.predict(desc.cpu().numpy())
    hist = np.bincount(labels, minlength=K).astype(np.float32)
    hist /= (hist.sum() + 1e-8)

    return torch.from_numpy(hist)

def build_dataset(samples, descriptors, kmeans, K):
    # Build dataset from pre-extracted descriptors
    X, y = [], []

    for i, s in enumerate(samples):
        # Encode descriptors
        hist = encode_descriptors(descriptors[i], kmeans, K)
        
        # Build dataset
        X.append(hist)
        y.append(s["label"])

    return torch.stack(X), torch.tensor(y)

def build_dataset_from_indices(samples, descriptors, indices, kmeans, K):
    # Build dataset from pre-extracted descriptors using specific indices
    X, y = [], []

    for idx in indices:
        # Encode descriptors
        hist = encode_descriptors(descriptors[idx], kmeans, K)
        
        # Build dataset
        X.append(hist)
        y.append(samples[idx]["label"])

    return torch.stack(X), torch.tensor(y)

In [17]:
# Subject-fold cross validation setup (custom splitter)
N_FOLDS = 5
RANDOM_SEED = 42

# Get unique person IDs for subject-based split
unique_persons = sorted(set(s["person"] for s in trainval_samples))
print(f"Unique persons in train+val: {unique_persons}")
print(f"Total persons: {len(unique_persons)}")

if N_FOLDS > len(unique_persons):
    raise ValueError(f"N_FOLDS={N_FOLDS} cannot be greater than number of persons={len(unique_persons)}")

# Create mapping from person to sample indices
person_to_indices = {}
for idx, sample in enumerate(trainval_samples):
    person = sample["person"]
    if person not in person_to_indices:
        person_to_indices[person] = []
    person_to_indices[person].append(idx)

def build_subject_folds(person_ids, n_folds=5, seed=42):
    person_ids = list(person_ids)
    rng = np.random.default_rng(seed)
    rng.shuffle(person_ids)

    fold_sizes = [len(person_ids) // n_folds] * n_folds
    for i in range(len(person_ids) % n_folds):
        fold_sizes[i] += 1

    folds = []
    start = 0
    for size in fold_sizes:
        folds.append(person_ids[start:start + size])
        start += size

    return folds

subject_folds = build_subject_folds(unique_persons, n_folds=N_FOLDS, seed=RANDOM_SEED)
print("Persons per fold:", [len(fold) for fold in subject_folds])

# Store results for each fold
fold_results = {
    'gnb': {'train_acc': [], 'val_acc': []},
    'mnb': {'train_acc': [], 'val_acc': []},
    'svm': {'train_acc': [], 'val_acc': []}
}

Unique persons in train+val: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Total persons: 21
Persons per fold: [5, 4, 4, 4, 4]


### 4. Train Classifier

In [18]:
def evaluate(clf, scaler, X, y):
    Xs = X if scaler is None else scaler.transform(X.numpy()) 
    y_pred = clf.predict(Xs)

    acc = balanced_accuracy_score(y.numpy(), y_pred)
    cm = confusion_matrix(y.numpy(), y_pred)

    return acc, cm

def train_svm(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = LinearSVC(C=1.0, max_iter=5000)
    clf.fit(Xs, y.numpy())

    return clf, scaler

def train_gnb(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = GaussianNB()
    clf.fit(Xs, y.numpy())
    
    return clf, scaler

class IntScaler():
    def __init__(self, scale=10000):
        self.scale = scale

    def fit_transform(self, data):
        return np.round(data * self.scale)
    
    def transform(self, data):
        return np.round(data * self.scale)

def train_mnb(X, y):
    scaler = IntScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = MultinomialNB()
    clf.fit(Xs, y.numpy())
    
    return clf, scaler

In [21]:
# Perform k-fold cross validation (subject-based split)
for fold_idx, val_persons in enumerate(subject_folds):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    val_persons = sorted(val_persons)
    train_persons = sorted([p for p in unique_persons if p not in set(val_persons)])

    print(f"Training persons: {train_persons}")
    print(f"Validation persons: {val_persons}")

    # Map persons to sample indices
    train_indices = []
    for person in train_persons:
        train_indices.extend(person_to_indices[person])

    val_indices = []
    for person in val_persons:
        val_indices.extend(person_to_indices[person])

    # Build vocabulary from training fold descriptors only
    train_fold_descriptors = [trainval_descriptors[i] for i in train_indices]
    print(f"Building vocabulary from {len(train_indices)} training samples...")
    kmeans_fold = build_vocabulary(train_fold_descriptors, K)

    # Build datasets for this fold
    X_train_fold, y_train_fold = build_dataset_from_indices(
        trainval_samples, trainval_descriptors, train_indices, kmeans_fold, K
    )
    X_val_fold, y_val_fold = build_dataset_from_indices(
        trainval_samples, trainval_descriptors, val_indices, kmeans_fold, K
    )

    print(f"Training set: {len(X_train_fold)} samples")
    print(f"Validation set: {len(X_val_fold)} samples")

    # Train classifiers
    print("Training classifiers...")
    clf_gnb, scaler_gnb = train_gnb(X_train_fold, y_train_fold)
    clf_mnb, scaler_mnb = train_mnb(X_train_fold, y_train_fold)
    clf_svm, scaler_svm = train_svm(X_train_fold, y_train_fold)

    # Evaluate on training fold
    train_acc_gnb, _ = evaluate(clf_gnb, scaler_gnb, X_train_fold, y_train_fold)
    train_acc_mnb, _ = evaluate(clf_mnb, scaler_mnb, X_train_fold, y_train_fold)
    train_acc_svm, _ = evaluate(clf_svm, scaler_svm, X_train_fold, y_train_fold)

    # Evaluate on validation fold
    val_acc_gnb, _ = evaluate(clf_gnb, scaler_gnb, X_val_fold, y_val_fold)
    val_acc_mnb, _ = evaluate(clf_mnb, scaler_mnb, X_val_fold, y_val_fold)
    val_acc_svm, _ = evaluate(clf_svm, scaler_svm, X_val_fold, y_val_fold)

    # Store results
    fold_results['gnb']['train_acc'].append(train_acc_gnb)
    fold_results['gnb']['val_acc'].append(val_acc_gnb)
    fold_results['mnb']['train_acc'].append(train_acc_mnb)
    fold_results['mnb']['val_acc'].append(val_acc_mnb)
    fold_results['svm']['train_acc'].append(train_acc_svm)
    fold_results['svm']['val_acc'].append(val_acc_svm)

    # Print fold results
    print(f"\nFold {fold_idx + 1} Results:")
    print(f"  Gaussian NB    - Train: {train_acc_gnb*100:.2f}%, Val: {val_acc_gnb*100:.2f}%")
    print(f"  Multinomial NB - Train: {train_acc_mnb*100:.2f}%, Val: {val_acc_mnb*100:.2f}%")
    print(f"  Linear SVM     - Train: {train_acc_svm*100:.2f}%, Val: {val_acc_svm*100:.2f}%")

# Print average k-fold results
print(f"\n\n{'='*60}")
print(f"K-FOLD CROSS VALIDATION RESULTS (averaged over {N_FOLDS} folds)")
print(f"{'='*60}")

for model_name, model_key in [('Gaussian NB', 'gnb'), ('Multinomial NB', 'mnb'), ('Linear SVM', 'svm')]:
    train_mean = np.mean(fold_results[model_key]['train_acc'])
    train_std = np.std(fold_results[model_key]['train_acc'])
    val_mean = np.mean(fold_results[model_key]['val_acc'])
    val_std = np.std(fold_results[model_key]['val_acc'])

    print(f"\n{model_name}:")
    print(f"  Train: {train_mean*100:.2f}% ± {train_std*100:.2f}%")
    print(f"  Val:   {val_mean*100:.2f}% ± {val_std*100:.2f}%")

# Train final models on ALL trainval data for test evaluation
print(f"\n\n{'='*60}")
print(f"TRAINING FINAL MODELS ON ALL TRAIN+VAL DATA")
print(f"{'='*60}")

print("Building vocabulary from all train+val data...")
kmeans_final = build_vocabulary(trainval_descriptors, K)
dump(kmeans_final, f"artifact/{PREFIX}kmeans_K{K}.joblib")

X_trainval, y_trainval = build_dataset(trainval_samples, trainval_descriptors, kmeans_final, K)
X_test, y_test = build_dataset(test_samples, test_descriptors, kmeans_final, K)

print(f"Training final classifiers on {len(X_trainval)} samples...")
clf_gnb_final, scaler_gnb_final = train_gnb(X_trainval, y_trainval)
clf_mnb_final, scaler_mnb_final = train_mnb(X_trainval, y_trainval)
clf_svm_final, scaler_svm_final = train_svm(X_trainval, y_trainval)


FOLD 1/5
Training persons: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 15, 18, 19, 20]
Validation persons: [11, 13, 16, 17, 21]
Building vocabulary from 4270 training samples...
Training set: 4270 samples
Validation set: 1354 samples
Training classifiers...

Fold 1 Results:
  Gaussian NB    - Train: 76.55%, Val: 62.20%
  Multinomial NB - Train: 73.43%, Val: 60.60%
  Linear SVM     - Train: 100.00%, Val: 67.92%

FOLD 2/5
Training persons: [1, 2, 3, 4, 5, 6, 9, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21]
Validation persons: [7, 8, 10, 15]
Building vocabulary from 4556 training samples...
Training set: 4556 samples
Validation set: 1068 samples
Training classifiers...

Fold 2 Results:
  Gaussian NB    - Train: 77.07%, Val: 59.57%
  Multinomial NB - Train: 72.95%, Val: 61.56%
  Linear SVM     - Train: 100.00%, Val: 75.74%

FOLD 3/5
Training persons: [2, 3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21]
Validation persons: [1, 4, 6, 18]
Building vocabulary from 4546 training samples...
Tra

### 5. Model Evaluation

In [22]:
def save_results(K, actions, n_folds, fold_results, test_accs, filename="results.txt", description=""):
    with open(filename, 'w') as f:
        f.write("="*60 + "\n")
        f.write("MODEL PERFORMANCE SUMMARY\n")
        f.write("="*60 + "\n")
        f.write(f"Description: {description}\n")
        f.write(f"K: {K}\n")
        f.write(f"Actions: {', '.join(actions)}\n")
        f.write(f"Cross Validation: {n_folds}-fold\n")
        f.write("-"*60 + "\n\n")
        
        f.write(f"{n_folds}-FOLD CROSS VALIDATION RESULTS\n")
        f.write("-"*60 + "\n")
        for model_name, model_key in [('Gaussian NB', 'gnb'), ('Multinomial NB', 'mnb'), ('Linear SVM', 'svm')]:
            train_mean = np.mean(fold_results[model_key]['train_acc'])
            train_std = np.std(fold_results[model_key]['train_acc'])
            val_mean = np.mean(fold_results[model_key]['val_acc'])
            val_std = np.std(fold_results[model_key]['val_acc'])
            f.write(f"\n{model_name}:\n")
            f.write(f"  Train: {train_mean*100:6.2f}% ± {train_std*100:5.2f}%\n")
            f.write(f"  Val:   {val_mean*100:6.2f}% ± {val_std*100:5.2f}%\n")
        
        f.write("\n" + "-"*60 + "\n\n")
        f.write("FINAL TEST ACCURACY (trained on all train+val)\n")
        f.write("-"*60 + "\n")
        for model_name, acc in test_accs.items():
            f.write(f"  {model_name:20s}: {acc*100:6.2f}%\n")
        f.write("\n" + "="*60 + "\n")
    
    print(f"Results saved to {filename}")

In [23]:
# Evaluate final models on test set
print(f"\n{'='*60}")
print(f"FINAL TEST SET EVALUATION")
print(f"{'='*60}\n")

test_acc_gnb, cm_gnb = evaluate(clf_gnb_final, scaler_gnb_final, X_test, y_test)
print(f"Gaussian Naive Bayes test accuracy: {test_acc_gnb*100:.2f}%")
test_acc_mnb, cm_mnb = evaluate(clf_mnb_final, scaler_mnb_final, X_test, y_test)
print(f"Multinomial Naive Bayes test accuracy: {test_acc_mnb*100:.2f}%")
test_acc_svm, cm_svm = evaluate(clf_svm_final, scaler_svm_final, X_test, y_test)
print(f"Linear SVM test accuracy: {test_acc_svm*100:.2f}%")

# Save all results
test_accs = { 'Gaussian NB': test_acc_gnb, 'Multinomial NB': test_acc_mnb, 'Linear SVM': test_acc_svm}
save_results(K, ACTIONS, N_FOLDS, fold_results, test_accs, f"artifact/{PREFIX}results_K{K}.txt", description=DESCRIPTION)


FINAL TEST SET EVALUATION

Gaussian Naive Bayes test accuracy: 73.13%
Multinomial Naive Bayes test accuracy: 74.52%
Linear SVM test accuracy: 77.10%
Results saved to artifact/0229-results_K800.txt
